# Init modules

In [0]:
import sys
sys.path.append("../..")

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

from common.config import bronze_table, silver_table
from common.io import read_table, write_silver
import common.transformations as TR

In [0]:
RENAME_MAP = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'firstname',
    'cst_lastname': 'lastname',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'create_date'
}

MARITAL_STATUS_MAP = {
    'S': 'Single',
    'M': 'Married'
}

GENDER_MAP = {
    'M': 'Male',
    'F': 'Female'
}

# Reading From Bronze

In [0]:
df = read_table(spark, bronze_table('crm_cust_info'))
df.display()

# Data Transformations

## Trim the string values

In [0]:
df = TR.trim_string_columns(df)
df.display()

## Normalization for martial_status, gndr

In [0]:
df = TR.map_codes_to_labels(df, 'cst_marital_status', MARITAL_STATUS_MAP)
df = TR.map_codes_to_labels(df, 'cst_gndr', GENDER_MAP)
df.display()

## Rename not friendly column names

In [0]:
df = TR.rename_columns(df, RENAME_MAP)
df.display()


# Write Into Silver Table

In [0]:
write_silver(df, silver_table('crm_customers'))